In [1]:
from model.LightGCN import *
from preprocess.AmazonBook import *
from evaluation import *
from util.dataset_splitter import DatasetSplitter
pd.options.display.max_rows = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path = './dataset/amazon-book'
dataset = AmazonBook(path)

data = dataset.get()
num_users, num_books = dataset.getNumber()
config = {
    'k': 20,
    'lr': 0.001,
    'epochs': 10,
    'num_layers': 2,
    'batch_size': 8194,
    'embedding_dim': 64,
    'num_users': num_users,
    'num_books': num_books,
    'tuning_type': None,
}
model = LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device)

# Split the dataset

In [2]:
# Basic setting
num_edges = data.edge_index.size(1)
perm = torch.randperm(num_edges)
split = int(num_edges * 0.1)  # 10% of edges will be retained

# Define the data for forget and retain dataset
forget_data = Data()
retain_data = Data()

forget_data.num_nodes = data.num_nodes
retain_data.num_nodes = data.num_nodes
forget_data.edge_index = data.edge_index[:, perm[:split]]
retain_data.edge_index = data.edge_index[:, perm[split:]]
forget_data.edge_label_index = data.edge_label_index
retain_data.edge_label_index = data.edge_label_index

datasplit = DatasetSplitter(dataset_name="AmazonBook")
datasplit.save_split(retain_data, forget_data)  

# Retain LightGCN model

In [ ]:
retrain_lightgcn = LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device)
retrain_lightgcn = lightgcn_evaluation(retrain_lightgcn, config, retain_data, device)

In [ ]:
# Test on the forget data
prompt_lightgcn_unlearning_evaluation(retrain_lightgcn, None, forget_data, config, device)

# Prompt Unlearning
## Case 1: Without Contrastive Loss

In [4]:
# Define the model
teacher = LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device)
student = LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device)

# Load the model
teacher.load_state_dict(torch.load(f"pretrain/LightGCN_Amazon_Book_{config['epochs']}_Epochs_Top_{config['k']}.pt"))
student.load_state_dict(torch.load(f"pretrain/LightGCN_Amazon_Book_{config['epochs']}_Epochs_Top_{config['k']}.pt"))

<All keys matched successfully>

In [5]:
# Setting the basic hyperparameters
config['alpha'] = 0.3
config['beta'] = 0.7
config['epochs'] = 10
config['gamma'] = 1e-6 # contrastive loss
config['tuning_type'] = 'simplePrompt' 
config['weight_decay'] = 0.001
config['Contrastive_loss'] = False
student, prompt = prompt_lightgcn_evaluation(teacher, student, data, retain_data, forget_data, config, device)

c:\Users\MSI\.conda\envs\master\lib\site-packages\torch_geometric\data\storage.py:450: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index', 'edge_label_index', 'x'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  warnings.warn(


Epoch: 001, Loss: 0.6972, HR@20: 0.0074, Recall@20: 0.0161, NDCG@20: 0.0129
Epoch: 002, Loss: 0.6949, HR@20: 0.0075, Recall@20: 0.0163, NDCG@20: 0.0131
Epoch: 003, Loss: 0.6914, HR@20: 0.0075, Recall@20: 0.0165, NDCG@20: 0.0132
Epoch: 004, Loss: 0.6866, HR@20: 0.0076, Recall@20: 0.0165, NDCG@20: 0.0133
Epoch: 005, Loss: 0.6807, HR@20: 0.0076, Recall@20: 0.0165, NDCG@20: 0.0134
Epoch: 006, Loss: 0.6737, HR@20: 0.0077, Recall@20: 0.0166, NDCG@20: 0.0134
Epoch: 007, Loss: 0.6661, HR@20: 0.0077, Recall@20: 0.0166, NDCG@20: 0.0135
Epoch: 008, Loss: 0.6580, HR@20: 0.0077, Recall@20: 0.0165, NDCG@20: 0.0135
Epoch: 009, Loss: 0.6498, HR@20: 0.0077, Recall@20: 0.0166, NDCG@20: 0.0135
Epoch: 010, Loss: 0.6417, HR@20: 0.0077, Recall@20: 0.0165, NDCG@20: 0.0135
Total time: 445.87s


In [6]:
# Test on the forget data
prompt_lightgcn_unlearning_evaluation(student, prompt, data, forget_data, config, device)

HR@20: 0.0063, Recall@20: 0.0309, NDCG@20: 0.0183


## Case 2: With Contrastive Loss

In [ ]:
# Setting the basic hyperparameters
config['Contrastive_loss'] = True

student, prompt = prompt_lightgcn_evaluation(teacher, student, data, retain_data, forget_data, config, device)

In [ ]:
# Test on the forget data
prompt_lightgcn_unlearning_evaluation(student, prompt, data, forget_data, config , device)

# Additional Materials(Multiple Prompts)

In [ ]:
config['tuning_type'] = 'complexPrompt'
config["number_p"] = 10 # The number of prompts
# Setting the basic hyperparameters
config['contrastive_loss'] = True

student, prompt = prompt_lightgcn_evaluation(teacher, student, data, retain_data, forget_data, config, device)

In [ ]:
# Test on the forget data
prompt_lightgcn_unlearning_evaluation(student, prompt, data, forget_data, config, device)